# RSNA Knee Abnormality Detection - CPU baseline & sample submission

- **Task:** predict the per-study probability of **12 knee MRI findings** (ACL, MCL, meniscus, OA compartments, effusion, etc.).
- **Metric:** macro-averaged ROC AUC over the twelve labels.
- **Submission:** one row per test study, confidence score per label -> file named `submission.csv`.
- **This notebook (v2):** runs on **CPU only**, keeps the data download **minimal** (only five tiny CSV files, *not* the ~570 GB of DICOMs), and upgrades the image model to **plane-aware preprocessing**: each finding is scored by a Tsetlin Machine trained on a middle slice from the anatomically preferred series (e.g. ACL/MCL/menisci on sagittal, OA on coronal, PF OA on axial), at `TM_IMG_SIZE=32`, with more binary feature maps (Otsu, adaptive mean, Canny, Sobel, thermometers).

### How to run
- **Kaggle Code (recommended):** add the competition dataset, pick CPU (or GPU) accelerator, run. Data is already mounted at `/kaggle/input` -> nothing is downloaded.
- **Offline-safe:** this competition disables internet during scoring, so the notebook never runs `pip install`; it only uses packages from the Kaggle base image (numpy, pandas, pydicom, matplotlib). DICOMs that stock `pydicom` cannot decode (JPEG2000 / JPEG-LS) are skipped automatically.
- **Locally:** install `kaggle` CLI with credentials (`kaggle competitions download`), then just run the cells. Only the 5 small CSVs are fetched.

The notebook builds two CPU baselines and writes a valid `submission.csv`:
1. **Baseline A** - predict each label at its training-set prevalence (guaranteed-valid scaffold, macro-AUC ~0.5).
2. **Baseline B** - a tiny logistic-regression model on study-level MRI *series descriptors* (planes, fluid sensitivity, fat suppression). Cross-validated on the train set, this is a real (if weak) learned baseline that beats 0.5.

On Kaggle (where the MRI images are mounted) a third model runs:
3. **Composite Tsetlin Machine** - several image-preprocessing methods each feed their own Tsetlin Machine; per-finding confidences come from summed, range-normalized **class_sums** (composite scoring as in the CAIR toolbox). CPU-only, so it fits the CPU track.


## 1. Data description (summary)

Each **study** = one knee MRI exam = several **series** (DICOM sequences). ~5,000 training studies, ~1,300 test studies.

| File | Contents |
|---|---|
| `train.csv` | one row per study: `StudyInstanceUID`, `PatientSex`, free-text `Report` (multilingual), 12 binary labels |
| `train_series.csv` | one row per series: study/series UIDs, `Fluid_Sensitive`, `Fat_Suppression`, `Anatomical_Plane` |
| `train_series/` | DICOMs `train_series/<Study>/<Series>/<Slice>.dcm` (20-45 slices per series) |
| `test.csv` / `test_series.csv` / `test_series/` | same, for ~1,300 test studies |
| `sample_submission.csv` | all labels set to `0.5` |

The 12 labels: `ACL`, `MCL`, `Medial Meniscus`, `Lateral Meniscus`, `Medial OA`, `Lateral OA`, `PF OA`, `Effusion`, `Synovitis`, `Baker's`, `Contusion`, `Fracture`.


In [ ]:
import os
import sys
import subprocess

import numpy as np
import pandas as pd

COMPETITION = "rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
          "Medial OA", "Lateral OA", "PF OA", "Effusion",
          "Synovitis", "Baker's", "Contusion", "Fracture"]

KAGGLE_INPUT = "/kaggle/input/competitions"
ON_KAGGLE = os.path.exists(KAGGLE_INPUT)
if ON_KAGGLE:
    # Find the mounted competition dataset by its files, not by a fixed slug,
    # and never write to the read-only input mount.
    _inputs = sorted(os.listdir(KAGGLE_INPUT))
    DATA_DIR = next(
        (os.path.join(KAGGLE_INPUT, d) for d in _inputs
         if os.path.isdir(os.path.join(KAGGLE_INPUT, d))
         and os.path.isfile(os.path.join(KAGGLE_INPUT, d, "train.csv"))),
        os.path.join(KAGGLE_INPUT, _inputs[0]) if _inputs else os.path.join(KAGGLE_INPUT, COMPETITION),
    )
    if not os.path.isfile(os.path.join(DATA_DIR, "train.csv")):
        raise SystemExit("Competition data not mounted. Add dataset '" + COMPETITION
                         + "' as an input to this notebook, then run again.")
else:
    DATA_DIR = os.path.join(os.getcwd(), "data")
    os.makedirs(DATA_DIR, exist_ok=True)

BASELINE_B_OK = False  # set to True if Baseline B trains successfully

print("Running on Kaggle:", ON_KAGGLE)
print("Data dir:", DATA_DIR)
print("Selected accelerator:", os.environ.get("KAGGLE_GPU_TYPE", "n/a (CPU)"))


## 2. Download the minimal data (CSVs only)

We deliberately **skip the ~570 GB of DICOM files**. This baseline needs only the five CSVs
(a few MB in total). On Kaggle they are already mounted, so nothing is downloaded there.


In [ ]:
MINIMAL_FILES = ["train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv"]

def ensure_minimal_data():
    if ON_KAGGLE:
        print("Data is mounted on Kaggle; skipping download.")
        return
    missing = [f for f in MINIMAL_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
    if not missing:
        print("All minimal files already present:", DATA_DIR)
        return
    print("Downloading (small) missing files:", missing)
    for f in missing:
        try:
            res = subprocess.run(
                ["kaggle", "competitions", "download", "-c", COMPETITION, "-f", f, "-p", DATA_DIR],
                check=True, capture_output=True, text=True,
            )
            print("  downloaded:", f)
        except Exception as e:
            print(f"  FAILED: {f} -> {e}")
            print("  Place the files manually into", DATA_DIR, "and re-run this cell.")
            raise SystemExit("Missing competition CSVs. See message above.")

ensure_minimal_data()


In [ ]:
def load(path):
    return pd.read_csv(path, dtype={"StudyInstanceUID": str})

train = load(os.path.join(DATA_DIR, "train.csv"))
train_series = load(os.path.join(DATA_DIR, "train_series.csv"))
test = load(os.path.join(DATA_DIR, "test.csv"))
test_series = load(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub = load(os.path.join(DATA_DIR, "sample_submission.csv"))


## 3. Quick look at the data

Only a small subset of training studies carries the 12 per-condition labels; the rest have a radiology `Report` you can mine for weak labels later.


In [ ]:
print("train:", train.shape, "| test:", test.shape)
print("train_series:", train_series.shape, "| test series/study approx:", train_series["StudyInstanceUID"].nunique())
print()
print("sample_submission columns:", list(sample_sub.columns))
print()
print("Are all 12 labels present in train.csv?", all(c in train.columns for c in LABELS))
print()
print("Labeled subset prevalence (positive rate):")
prev = train[LABELS].mean()
prev.to_frame("prevalence").round(3)


## 4. Baseline A - predict the training prevalence (sample submission)

For each label, assign its observed positive rate in the labeled training subset to every test study.
This produces a **valid submission** but, being constant per label, it cannot rank studies (macro-AUC ~ 0.5).
Use it as the sanity-check scaffold that the pipeline, formatting, and `submission.csv` are correct.


In [ ]:
prevalence = train[LABELS].mean()

submission_a = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"]})
for lab in LABELS:
    submission_a[lab] = prevalence[lab]

print("Baseline A built for", len(submission_a), "test studies.")
submission_a.head()


## 5. Weak labels from the radiology reports (the real lever)

Only a small subset of the ~5,000 training studies carry expert labels - the rest only have the
multilingual radiology `Report` text. We recover labels for nearly all of them two ways:

1. **Keyword mining** - per-finding phrase patterns (EN/ES/FR/DE/PT) with negation handling
   ("no ACL tear", "medial meniscus is intact") give high-precision weak labels.
2. **Character n-gram TF-IDF + logistic regression** (sklearn, no internet) - a language-agnostic
   model fit on the expert-labeled reports that fills any study a keyword misses.

Weak probabilities are binarized at 0.5 and merged into `train[LABELS]`: expert labels win where
present, weak labels fill the rest. This turns a few-hundred-study problem into a ~5,000-study one.
Test studies have no reports, so this widens *training* supervision only.


In [ ]:
import re as _re

_NEG_WORDS = _re.compile(
    r"(?i)\b(no|without|absent|intact|normal|unremarkable|negative|not)\b"
    r"|no evidence of|no sign of|no definite|no findings of|rules? out|no radiologic evidence")
_MILD_WORDS = _re.compile(r"(?i)\b(mild|minimal|leve|light)\b")

POS_PATTERNS = {
    "ACL": [
        r"\bacl\b\s*(?:tear|injury|rupture|sprain|strain|disruption|deficiency|reconstruction|graft|torn)",
        r"anterior cruciate\s*(?:ligament)?\s*(?:tear|injury|rupture|sprain|strain|disruption|deficiency|reconstruction|graft)",
        r"tear(?:s)? of the acl",
    ],
    "MCL": [
        r"\bmcl\b\s*(?:tear|injury|rupture|sprain|strain|disruption|avulsion|torn)",
        r"medial collateral\s*(?:ligament)?\s*(?:tear|injury|rupture|sprain|strain|disruption|avulsion|torn)",
        r"tear(?:s)? of the (?:mcl|medial collateral)",
    ],
    "Medial Meniscus": [
        r"medial\s+menisc(?:al|us)\s+(?:tear|injury|rupture|flap|displac|torn)",
        r"tear(?:s)? of the medial menisc(?:al|us)",
        r"torn medial menisc(?:al|us)",
    ],
    "Lateral Meniscus": [
        r"lateral\s+menisc(?:al|us)\s+(?:tear|injury|rupture|flap|displac|torn)",
        r"tear(?:s)? of the lateral menisc(?:al|us)",
        r"torn lateral menisc(?:al|us)",
    ],
    "Medial OA": [
        r"medial\s+joint space narrowing",
        r"medial\s+(?:compartment|tibiofemoral|femorotibial)\s+joint space narrowing",
        r"medial\s+(?:compartment|tibiofemoral|femorotibial)\s+narrowing",
        r"medial\s+(?:compartment\s+)?(?:osteoarthritis|osteoarthrosis|osteoarthritic changes|degenerative changes)",
        r"medial\s+osteophyt[ae]s?",
        r"medial\s+compartment\s+cartilage loss",
        r"medial\s+(?:compartment\s+)?bone[ -]on[ -]bone",
        r"loss of medial joint space",
    ],
    "Lateral OA": [
        r"lateral\s+joint space narrowing",
        r"lateral\s+(?:compartment|tibiofemoral|femorotibial)\s+joint space narrowing",
        r"lateral\s+(?:compartment|tibiofemoral|femorotibial)\s+narrowing",
        r"lateral\s+(?:compartment\s+)?(?:osteoarthritis|osteoarthrosis|osteoarthritic changes|degenerative changes)",
        r"lateral\s+osteophyt[ae]s?",
        r"lateral\s+compartment\s+cartilage loss",
        r"lateral\s+(?:compartment\s+)?bone[ -]on[ -]bone",
        r"loss of lateral joint space",
    ],
    "PF OA": [
        r"patello[ -]?femoral\s+joint space narrowing",
        r"patello[ -]?femoral\s+(?:osteoarthritis|osteoarthrosis|osteoarthritic changes|degenerative changes|narrowing|joint space loss|osteophyt[ae]s?|cartilage loss|sclerosis|bone[ -]on[ -]bone|chondromalacia)",
        r"retropatellar\s+(?:cartilage loss|osteophyt[ae]s?|degeneration|narrowing|sclerosis|chondromalacia)",
        r"trochlear\s+(?:cartilage loss|osteophyt[ae]s?|degenerative changes|narrowing|sclerosis|chondromalacia)",
        r"\bpfj\b\s+(?:narrowing|osteoarthritis|degenerative|osteophyt[ae]s?|joint space)",
    ],
    "Effusion": [
        r"\beffusion\b", r"joint effusion", r"intra[ -]articular fluid",
        r"derrame", r"epanchement", r"\u00e9panchement", r"erguss",
    ],
    "Synovitis": [
        r"synovitis", r"sinovitis",
        r"synovial\s+(?:thickening|enhancement|proliferation|inflammation|hypertrophy)",
        r"pannus",
    ],
    "Baker's": [
        r"baker[\'’]?s?\s+cyst", r"popliteal cyst", r"quiste de baker", r"backercyst",
    ],
    "Contusion": [
        r"bone\s+(?:marrow\s+)?(?:contusion|edema|bruise)",
        r"marrow\s+edema",
        r"bone\s+bruis(?:e|ing)",
        r"contusi[o\u00f3]n", r"edema\u00b3seo", r"bone bruising",
    ],
    "Fracture": [
        r"\bfracture\b", r"fractures", r"fractura", r"fraktur",
        r"stress fracture", r"avulsion fracture", r"tibial plateau fracture",
    ],
}

def _prev_window(text, start, n=28):
    seg = text[max(0, start - n):start]
    brk = max(seg.rfind("."), seg.rfind(";"), seg.rfind(":"),
              seg.rfind("("), seg.rfind(")"), seg.rfind("\n"))
    return seg[brk + 1:] if brk >= 0 else seg

def keyword_score(text, label):
    t = _re.sub(r"\d+", " ", text.lower())
    scores = []
    for pat in POS_PATTERNS[label]:
        for m in _re.finditer(pat, t):
            prev = _prev_window(t, m.start())
            nxt = t[m.end():m.end() + 14]
            if _NEG_WORDS.search(prev + " " + nxt):
                scores.append(0.05)
            elif label.endswith("OA") and _MILD_WORDS.search(prev):
                scores.append(0.6)
            else:
                scores.append(0.95)
    if not scores:
        return None
    return min(scores) if any(s <= 0.05 for s in scores) else max(scores)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def cv_auc(clf, X, y, n_splits=5, seed=42):
    y = pd.Series(y).reset_index(drop=True)
    pos = int(y.sum())
    if pos < 2 or len(y) - pos < 2:
        return np.nan
    n_splits = min(n_splits, pos, len(y) - pos)
    if n_splits < 2:
        return np.nan
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr, va in skf.split(np.zeros(len(y)), y):
        if hasattr(X, "iloc"):
            Xtr, Xva = X.iloc[tr], X.iloc[va]
        else:
            Xtr, Xva = X[tr], X[va]
        clf.fit(Xtr, y.iloc[tr])
        oof[va] = clf.predict_proba(Xva)[:, 1]
    return roc_auc_score(y, oof)

train = train.copy()
if "Report" in train.columns:
    train["_text"] = train["Report"].map(
        lambda s: _re.sub(r"\d+", " ", str(s).lower()) if isinstance(s, str) else "")
else:
    train["_text"] = ""
_have_text = train["_text"].str.strip() != ""
if not _have_text.any():
    print("  NOTE: no Report text found - skipping text weak labels, expert labels only.")

# --- 1) keyword weak labels ---
kw_prob = pd.DataFrame(index=train.index, columns=LABELS)
for lab in LABELS:
    kw_prob[lab] = train["_text"].map(
        lambda t: keyword_score(t, lab) if t.strip() else None)

# --- 2) language-agnostic char n-gram TF-IDF + logistic regression ---
text_prob = pd.DataFrame(np.nan, index=train.index, columns=LABELS)
cv_text = {}
Xrep = None
if _have_text.any():
    try:
        _vect = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4),
                                min_df=2, sublinear_tf=True)
        Xrep = _vect.fit_transform(train.loc[_have_text, "_text"])
    except Exception as e:
        print("  text vectorizer failed:", e)
        Xrep = None

if Xrep is not None:
    for lab in LABELS:
        lab_sel = train[lab].notna() & _have_text
        yy = train[lab][lab_sel]
        if yy.nunique() < 2 or int(yy.sum()) < 2 or (len(yy) - int(yy.sum())) < 2:
            continue
        Xy = Xrep[list(np.where(lab_sel.values)[0])]
        cv_text[lab] = cv_auc(_LR(C=3.0, max_iter=2000), Xy, yy)
        clf = _LR(C=3.0, max_iter=2000).fit(Xy, yy)
        text_prob.loc[_have_text, lab] = clf.predict_proba(Xrep)[:, 1]
print("text model CV AUC:", pd.Series(cv_text).round(3).to_dict())

# --- merge: expert labels win, weak labels fill the rest ---
orig_expert = train[LABELS].notna()
train["_expert"] = orig_expert.all(axis=1)

weak_prob = kw_prob.where(kw_prob.notna(), text_prob)
for lab in LABELS:
    w = weak_prob[lab]
    need = w.notna() & train[lab].isna()
    train.loc[need, lab] = (w[need] >= 0.5).astype(float)

_weak_coverage = float(train[LABELS].notna().mean().mean())
train["_weak_conf"] = weak_prob.sub(0.5).abs().fillna(0.0).mean(axis=1)
train.loc[train["_expert"], "_weak_conf"] = 1.0

n_full = int(train[LABELS].notna().all(axis=1).sum())
print("studies fully labeled: %d / %d  (was %d expert)"
      % (n_full, len(train), int(orig_expert.all(axis=1).sum())))
print("weak-label coverage of study x label matrix: %.1f%%" % (100.0 * _weak_coverage))


## 6. Baseline B - tiny CPU model on MRI series descriptors + patient sex

`train_series.csv` tells us *which* MRI sequences each study contains (fluid-sensitive?
fat-suppressed? which anatomical plane?) and `train.csv` gives patient sex. Small **logistic
regression + gradient-boosted** models on those counts can rank studies and beat 0.5 macro-AUC.

Since section 5 filled report-derived labels for unlabeled studies, this baseline now trains on the
augmented ~5,000-study label set instead of just the expert-labeled subset. Each label combines a
scaled logistic regression with a histogram gradient boosting classifier.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

def series_features(series_df):
    uid = series_df["StudyInstanceUID"]
    feats = pd.DataFrame({
        "n_series": series_df.groupby(uid).size(),
        "n_fluid": series_df.groupby(uid)["Fluid_Sensitive"].sum(),
        "n_fatsat": series_df.groupby(uid)["Fat_Suppression"].sum(),
    })
    feats["n_nonfluid"] = feats["n_series"] - feats["n_fluid"]
    planes = pd.get_dummies(series_df["Anatomical_Plane"], prefix="plane")
    planes[uid.name] = series_df["StudyInstanceUID"].values
    plane_counts = planes.groupby(uid.name).sum()
    feats = feats.join(plane_counts, how="outer").fillna(0)
    for p in ["plane_Sagittal", "plane_Coronal", "plane_Axial"]:
        if p not in feats:
            feats[p] = 0
    feats["has_fluid"] = (feats["n_fluid"] > 0).astype(int)
    feats["has_fatsat"] = (feats["n_fatsat"] > 0).astype(int)
    feats["frac_fluid"] = (feats["n_fluid"] / feats["n_series"].replace(0, np.nan)).fillna(0)
    feats["frac_fatsat"] = (feats["n_fatsat"] / feats["n_series"].replace(0, np.nan)).fillna(0)
    return feats

X_train = series_features(train_series)
X_test = series_features(test_series)

# Align to the study tables (a few studies may lack series rows -> zeros).
X_train = X_train.reindex(train["StudyInstanceUID"]).fillna(0)
X_test = X_test.reindex(test["StudyInstanceUID"]).fillna(0)

if "PatientSex" in train.columns:
    sex_map = train.set_index("StudyInstanceUID")["PatientSex"].map({"Male": 1, "Female": 0})
    X_train["sex"] = sex_map.reindex(X_train.index).fillna(0.5)
else:
    X_train["sex"] = 0.5
X_test["sex"] = 0.5

y_train = train.set_index("StudyInstanceUID")[LABELS].reindex(X_train.index)

cv_scores = {}
pred_test = pd.DataFrame(index=X_test.index)
for lab in LABELS:
    y = y_train[lab]
    labeled = y.notna()
    if labeled.sum() < 2 or y[labeled].nunique() < 2:
        print(f"  {lab}: too few labeled examples -> falling back to prevalence")
        pred_test[lab] = prevalence[lab]
        cv_scores[lab] = np.nan
        continue
    cv_scores[lab] = cv_auc(LogisticRegression(max_iter=1000), X_train[labeled], y[labeled])
    scaler = StandardScaler().fit(X_train[labeled])
    lr = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train[labeled]), y[labeled])
    hgb = HistGradientBoostingClassifier(random_state=42, max_iter=200).fit(X_train[labeled], y[labeled])
    p_lr = lr.predict_proba(scaler.transform(X_test))[:, 1]
    p_hgb = hgb.predict_proba(X_test)[:, 1]
    pred_test[lab] = (pd.Series(p_lr).rank(pct=True) + pd.Series(p_hgb).rank(pct=True)).values / 2.0

cv_series = pd.Series(cv_scores, name="CV AUC")
print(cv_series.round(3).to_frame())
print("macro CV AUC:", round(float(np.nanmean(cv_series.values)), 4))
BASELINE_B_OK = True


## 7. Composite Tsetlin Machine model on MRI slices (runs on Kaggle)

This follows the "Composite Tsetlin Machine" approach from
[cair/An-Optimized-Toolbox-for-Advanced-Image-Processing-with-Tsetlin-Machine-Composites](https://github.com/cair/An-Optimized-Toolbox-for-Advanced-Image-Processing-with-Tsetlin-Machine-Composites).

Several image-preprocessing *methods* (Otsu thresholding, adaptive-mean thresholding, edge detection) each feed their
own Tsetlin Machine. For every finding we keep the positive-class **class_sums** each method's TM produces and combine
them exactly like `CIFAR10figure.py` does:

```
votes += class_sums / np.ptp(class_sums)     # range-normalized class sums
```

The summed votes are min-max scaled to `[0, 1]` to give the submission confidence per label (ranking, hence macro-AUC,
is unaffected by this monotone transform).

Tsetlin Machines are CPU algorithms, so this fits the CPU track. DICOMs are only available on Kaggle, so this section
runs there automatically and is skipped locally (keeping the download minimal). The TM code is self-contained
(numpy only) because Kaggle disables internet during scoring - no external package installs are possible.

The image model is trained on the **augmented** label set from section 5 (expert labels plus report-derived
weak labels), so it sees far more studies than the expert-labeled subset alone.


### 7.1 Settings

Plane-aware, 32px CPU-friendly defaults. Series are selected per finding via `train_series.csv`
(anatomical plane + fluid sensitivity); `TM_IMG_SIZE=32` gives 4x more spatial detail than v1.
Raise these to scale quality (at the cost of runtime).


In [ ]:
# --- Composite TM settings (image-based; requires mounted DICOMs on Kaggle) ---
TM_RUN = ON_KAGGLE        # set True to enable (needs train_series/test_series images)
TM_N_TRAIN = 1500         # max studies used for training (expert first, then report-derived weak)
TM_EPOCHS = 50
TM_CLAUSES = 150
TM_T = 800                # threshold parameter (number of states)
TM_S = 3.0                # specificity (feedback probability ~1/s)
TM_BOOST = 1              # boost true positive feedback (1 = pyTsetlinMachine default)
TM_MAX_INCLUDED = 8       # max included literals per clause; small keeps clauses sparse so they fire
TM_IMG_SIZE = 32          # middle-slice downsampled to IMG_SIZE x IMG_SIZE
TM_THERMO_BITS = 4        # thermometer levels (multi-threshold binarization per pixel)
TM_LABELS = LABELS
# Order in which series are preferred when selecting the image for a study.
TM_PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
# Preferred anatomical plane per finding (used when both planes are available).
TM_PLANE_OF_LABEL = {
    "ACL": "Sagittal", "MCL": "Sagittal",
    "Medial Meniscus": "Sagittal", "Lateral Meniscus": "Sagittal",
    "Medial OA": "Coronal", "Lateral OA": "Coronal", "PF OA": "Axial",
    "Effusion": "Sagittal", "Synovitis": "Sagittal",
    "Baker's": "Sagittal", "Contusion": "Sagittal", "Fracture": "Sagittal",
}
TM_METHODS = ["otsu", "adaptive_mean", "canny", "sobel", "thermo"]
print("TM image model enabled:", TM_RUN)


### 7.2 Self-contained Tsetlin Machine (numpy)

A self-contained numpy port of the pyTsetlinMachine v3.x binary TM (the algorithm behind the
CAIR composite toolbox): 2F literals, 8-bit thermometer TA states (literal included iff state
>= 128), even/odd clause-parity voting with clause weights, class_sum-gated Type Ia/Ib/II
feedback, and `max_included_literals` (kept small so clauses stay sparse and actually fire).
`predict_class_sums` returns the per-class vote matrix `(n_samples, 2)`, the same object the
toolbox scoring scripts save to disk.


In [ ]:
class BinaryTsetlinMachine:
    def __init__(self, clauses=150, T=800, s=3.0, epochs=20, boost_true_positive_feedback=1,
                 weighted_clauses=False, max_included_literals=None, seed=42):
        self.clauses = clauses
        self.T = int(T)
        self.s = s
        self.epochs = epochs
        self.boost = boost_true_positive_feedback
        self.weighted = weighted_clauses
        self.max_included = max_included_literals
        self.seed = seed
        self.state_bits = 8
        self.INC_THR = 2 ** (self.state_bits - 1)
        self.STATE_MAX = 2 ** self.state_bits - 1

    def _init(self, n_features):
        self.F = int(n_features)
        self.LIT = 2 * self.F
        rng = np.random.default_rng(self.seed)
        self._rng = np.random.default_rng(self.seed)
        # pyTsetlinMachine initializes every TA just below the inclusion threshold
        # (all lower thermometer bits set, top bit 0): all clauses start EMPTY and
        # fire on everything during training, which bootstraps learning.
        self.states = np.full((2, self.clauses, self.LIT), self.INC_THR - 1, dtype=np.int32)
        self.weights = np.ones((2, self.clauses), dtype=np.int32)
        if self.max_included is None:
            self.max_included = self.LIT

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        self._init(X.shape[1])
        for _ in range(self.epochs):
            for i in range(len(X)):
                self._update(X[i], y[i])
        return self

    @staticmethod
    def _literals(xi):
        return np.concatenate([xi, 1 - xi]).astype(np.int32)

    def _update(self, xi, y):
        L = self._literals(xi)
        p_lit = 1.0 / self.s
        for c in range(2):
            M = self.states[c] >= self.INC_THR
            counts = M.sum(axis=1)
            S = M.astype(np.int32) @ L
            out = (S == counts)                 # empty clauses fire during UPDATE
            sign = np.where(np.arange(self.clauses) % 2 == 0, 1, -1)
            class_sum = int((sign * self.weights[c] * out).sum())
            class_sum = int(np.clip(class_sum, -self.T, self.T))
            target = 1 if c == y else 0
            gate_p = (self.T + (1 - 2 * target) * class_sum) / (2.0 * self.T)
            gate = self._rng.random(self.clauses) <= gate_p
            for j in range(self.clauses):
                if not gate[j]:
                    continue
                typ = (2 * target - 1) * (1 - 2 * (j % 2))
                if typ == -1:                    # Type II clause
                    if out[j]:
                        if self.weighted and self.weights[c, j] > 1:
                            self.weights[c, j] -= 1
                        inc = (~L.astype(bool)) & (~M[j])   # false AND excluded
                        self.states[c, j, inc] += 1
                else:                            # Type I clause
                    if out[j] and counts[j] <= self.max_included:   # Type Ia
                        if self.weighted:
                            self.weights[c, j] += 1
                        active = self._rng.random(self.LIT) < p_lit
                        inc = L == 1
                        if self.boost != 1:
                            inc = inc & active
                        dec = (L == 0) & active
                        self.states[c, j, inc] += 1
                        self.states[c, j, dec] -= 1
                    elif out[j]:                                  # Type Ib
                        active = self._rng.random(self.LIT) < p_lit
                        self.states[c, j, active] -= 1
            self.states[c] = np.clip(self.states[c], 0, self.STATE_MAX)

    def predict_class_sums(self, X):
        X = np.asarray(X)
        n = len(X)
        L = np.concatenate([X, 1 - X], axis=1).astype(np.int32)   # (n, LIT)
        sums = np.zeros((n, 2), dtype=np.float64)
        sign = np.where(np.arange(self.clauses) % 2 == 0, 1.0, -1.0)
        for c in range(2):
            M = (self.states[c] >= self.INC_THR).astype(np.int32)
            counts = M.sum(axis=1)
            S = L @ M.T                                            # (n, C) true-literal counts
            out = (S == counts[None, :]) & (counts[None, :] > 0)   # empty clauses suppressed at PREDICT
            sums[:, c] = (out * sign[None, :] * self.weights[c][None, :]).sum(axis=1)
        return sums


In [ ]:
# --- sanity check: the TM must learn a simple spatial pattern (runs anywhere, no data) ---
rng = np.random.default_rng(0)
n = 120
side = 16
X_s = np.zeros((n, side * side), dtype=np.int8)
y_s = np.zeros(n, dtype=int)
for i in range(n):
    lab = i % 2
    y_s[i] = lab
    img = rng.random((side, side)) < 0.25
    img[:, 0 if lab == 0 else -1] = True   # vertical bar on the correct side
    X_s[i] = img.ravel().astype(np.int8)
tm_check = BinaryTsetlinMachine(clauses=150, T=800, s=3.0, epochs=50,
                                boost_true_positive_feedback=1, max_included_literals=8, seed=1)
tm_check.fit(X_s, y_s)
s_sums = tm_check.predict_class_sums(X_s)
acc = (s_sums.argmax(axis=1) == y_s).mean()
print("sanity check train accuracy: %.3f" % acc)


### 7.3 Slice loading and binary feature maps

For each study we read the **middle slice of one series per anatomical plane** (sagittal, coronal,
axial) and downsample it to `TM_IMG_SIZE`. Each preprocessing method then turns a slice into a binary
feature vector for a TM. Per finding the preferred plane (`TM_PLANE_OF_LABEL`) is picked when
available, falling back to any other readable series. All implementations are pure numpy.


In [ ]:
def downsample(img, size=16):
    h, w = img.shape
    s = min(h, w)
    if s < size:                       # upscale tiny slices (rare) to at least size x size
        r = int(np.ceil(size / s))
        img = np.repeat(np.repeat(img, r, axis=0), r, axis=1)
        h, w = img.shape
        s = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    img = img[y0:y0 + s, x0:x0 + s]
    s -= s % size                      # drop the remainder so s is an exact multiple of size
    if s < size:
        return None
    img = img[:s, :s]
    ph = pw = s // size
    return img.reshape(size, ph, size, pw).mean(axis=(1, 3))

def otsu_binary(img):
    hist, edges = np.histogram(img, bins=64, range=(0.0, 1.0))
    centers = 0.5 * (edges[:-1] + edges[1:])
    w = np.cumsum(hist)
    wb = hist.sum() - w
    m = np.cumsum(hist * centers)
    mg = m[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        mu0 = m / np.where(w == 0, 1, w)
        mu1 = (mg - m) / np.where(wb == 0, 1, wb)
        between = w * wb * (mu0 - mu1) ** 2
    thr = centers[np.nanargmax(between)]
    return img >= thr

def box_mean(img, k=5):
    H, W = img.shape
    ii = np.zeros((H + 1, W + 1))
    ii[1:, 1:] = np.cumsum(np.cumsum(img, axis=0), axis=1)
    half = k // 2
    bot = np.minimum(np.arange(H) + half + 1, H)
    top = np.maximum(np.arange(H) - half, 0)
    right = np.minimum(np.arange(W) + half + 1, W)
    left = np.maximum(np.arange(W) - half, 0)
    br = ii[np.ix_(bot, right)]; tr = ii[np.ix_(top, right)]
    bl = ii[np.ix_(bot, left)]; tl = ii[np.ix_(top, left)]
    sums = br - tr - bl + tl
    cnt = (bot - top)[:, None] * (right - left)[None, :]
    return sums / np.maximum(cnt, 1)

def adaptive_mean_binary(img, k=5):
    return img > box_mean(img, k)

def canny_binary(img):
    gx = np.zeros_like(img)
    gy = np.zeros_like(img)
    gx[:, 1:] = img[:, 1:] - img[:, :-1]
    gy[1:, :] = img[1:, :] - img[:-1, :]
    mag = np.hypot(gx, gy)
    thr = mag.mean() + 0.5 * mag.std()
    return mag > thr

def _conv2(img, kern):
    H, W = img.shape
    kh, kw = kern.shape
    ph, pw = kh // 2, kw // 2
    p = np.pad(img, ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros_like(img)
    for i in range(kh):
        for j in range(kw):
            out += kern[i, j] * p[i:i + H, j:j + W]
    return out

def sobel_binary(img):
    kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    ky = kx.T
    mag = np.hypot(_conv2(img, kx), _conv2(img, ky))
    thr = mag.mean() + 0.5 * mag.std()
    return mag > thr

def thermo_binary(img):
    k = TM_THERMO_BITS
    thrs = np.linspace(0.1, 0.9, k)
    return np.stack([img >= t for t in thrs], axis=-1).ravel()

TM_METHOD_FUNCS = {
    "otsu": otsu_binary,
    "adaptive_mean": adaptive_mean_binary,
    "canny": canny_binary,
    "sobel": sobel_binary,
    "thermo": thermo_binary,
}

def binary_features(img):
    return {name: fn(img).ravel() for name, fn in TM_METHOD_FUNCS.items()}

def read_middle_slice(series_dir):
    import glob as _glob
    import pydicom
    files = sorted(_glob.glob(os.path.join(series_dir, "*.dcm")))
    if not files:
        return None
    # Slice files are named by SOPInstanceUID, not slice order; sort by
    # InstanceNumber (headers only) and take the spatial middle slice.
    nums = []
    for f in files:
        try:
            hdr = pydicom.dcmread(f, stop_before_pixels=True, force=True)
            nums.append(int(getattr(hdr, "InstanceNumber", 0)))
        except Exception:
            nums.append(-1)
    order = np.argsort(nums)
    mid = files[order[len(order) // 2]]
    # Some DICOMs use JPEG2000 / JPEG-LS transfer syntaxes that stock pydicom
    # cannot decode without extra (uninstallable) libs - skip those studies.
    try:
        ds = pydicom.dcmread(mid, force=True)
        img = ds.pixel_array.astype(np.float32)
        if img.ndim == 3:                  # defensive: multi-frame pixel data
            img = img[img.shape[0] // 2]
    except Exception:
        return None
    lo, hi = np.percentile(img, [1, 99])
    if hi > lo:
        img = np.clip((img - lo) / (hi - lo), 0, 1)
    else:
        img = np.zeros_like(img)
    return downsample(img, TM_IMG_SIZE)

def build_study_images(uids, root, series_meta=None):
    # Plane-aware series picker. Returns {uid: {plane: middle-slice image}} using
    # train/test_series.csv metadata when available; falls back to slice-count order.
    # Reads at most one series per plane per study (keeps DICOM reading bounded).
    import glob as _glob
    imgs = {}
    meta = None
    if series_meta is not None:
        meta = series_meta.groupby("StudyInstanceUID")
        meta = {uid: g for uid, g in meta}
    for uid in uids:
        study_dir = os.path.join(root, uid)
        if not os.path.isdir(study_dir):
            continue
        series = sorted(
            [d for d in _glob.glob(os.path.join(study_dir, "*")) if os.path.isdir(d)],
            key=lambda d: len(_glob.glob(os.path.join(d, "*.dcm"))), reverse=True)
        by_plane = {}
        if meta is not None and uid in meta:
            for _, r in meta[uid].iterrows():
                sid = str(r["SeriesInstanceUID"])
                d = os.path.join(study_dir, sid)
                if os.path.isdir(d):
                    by_plane.setdefault(str(r["Anatomical_Plane"]), []).append(d)
        per_plane = {}
        for plane in TM_PLANE_ORDER:
            cands = by_plane.get(plane, []) or series
            for series_dir in cands:
                im = read_middle_slice(series_dir)
                if im is not None:
                    per_plane[plane] = im
                    break
        if not per_plane:                  # no metadata match: keep best slice-count series
            for series_dir in series:
                im = read_middle_slice(series_dir)
                if im is not None:
                    per_plane["Fallback"] = im
                    break
        if per_plane:
            imgs[uid] = per_plane
    return imgs

def plane_for(uid, lab, imgs):
    # Pick the anatomical plane to score `lab` for study `uid`.
    d = imgs.get(uid)
    if not d:
        return None
    pref = TM_PLANE_OF_LABEL.get(lab)
    if pref in d:
        return d[pref]
    return d.get("Fallback") or d[next(iter(d))]


### 7.4 Train the composite and predict test confidences

For each of the 12 findings a TM is trained per preprocessing method on the training set (expert
labels plus section-5 report-derived weak labels). The image for a study is chosen per finding from
the preferred plane when available (`plane_for`). Per-method positive-class `class_sums` are
range-normalized (`/ np.ptp`) and summed (the composite), then min-max scaled to `[0, 1]` as the
submission confidence. Studies without usable images keep the training prevalence.


In [ ]:
import time as _time

TM_OK = False
if TM_RUN:
    t0 = _time.time()

    labeled = train.dropna(subset=TM_LABELS)
    # Prioritize expert-labeled studies, then report-derived weak ones by confidence.
    tr_expert = [u for u in train.loc[train["_expert"], "StudyInstanceUID"]
                 if u in set(labeled["StudyInstanceUID"])]
    tr_weak = (labeled.loc[~labeled["_expert"], ["StudyInstanceUID", "_weak_conf"]]
               .sort_values("_weak_conf", ascending=False)["StudyInstanceUID"].tolist())
    tr_cand = (tr_expert + tr_weak)[:TM_N_TRAIN]
    print("training TM on up to", len(tr_cand), "studies",
          "(%d expert, %d report-derived)" % (len(tr_expert), max(0, len(tr_cand) - len(tr_expert))))

    tr_imgs = build_study_images(tr_cand, os.path.join(DATA_DIR, "train_series"), train_series)
    te_imgs = build_study_images(test["StudyInstanceUID"], os.path.join(DATA_DIR, "test_series"), test_series)
    print("train images:", len(tr_imgs), "| test images:", len(te_imgs),
          "in %.1fs" % (_time.time() - t0))

    tr_uid = [u for u in tr_cand if u in tr_imgs]
    te_uid = [u for u in test["StudyInstanceUID"] if u in te_imgs]
    te_pos = [list(test["StudyInstanceUID"]).index(u) for u in te_uid]

    if len(tr_uid) < 10 or len(te_uid) == 0:
        print("WARNING: not enough usable images - falling back to the CSV baselines.")
    else:
        pred_tm = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"].tolist()})
        for lab in TM_LABELS:
            default_rate = float(train[lab].mean()) if pd.notna(train[lab].mean()) else 0.5
            pred_tm[lab] = default_rate
            # Per-finding plane selection: one image per study from the preferred plane.
            tr_lab_uid = [u for u in tr_uid if plane_for(u, lab, tr_imgs) is not None]
            te_lab_uid = [u for u in te_uid if plane_for(u, lab, te_imgs) is not None]
            if len(tr_lab_uid) < 10 or len(te_lab_uid) == 0:
                print(f"  {lab}: too few usable images, using prevalence {default_rate:.3f}")
                continue
            ylab = labeled.set_index("StudyInstanceUID").reindex(tr_lab_uid)[lab]
            mask = ylab.notna().values
            yv = ylab.values[mask].astype(int)
            if yv.sum() < 2 or (1 - yv).sum() < 2:
                print(f"  {lab}: too few labeled examples, using prevalence {default_rate:.3f}")
                continue
            te_lab_pos = [list(test["StudyInstanceUID"]).index(u) for u in te_lab_uid]
            Xtr_feats = {m: np.stack([binary_features(plane_for(u, lab, tr_imgs))[m]
                                      for u in tr_lab_uid]).astype(np.int8) for m in TM_METHODS}
            Xte_feats = {m: np.stack([binary_features(plane_for(u, lab, te_imgs))[m]
                                      for u in te_lab_uid]).astype(np.int8) for m in TM_METHODS}
            votes = np.zeros(len(te_lab_uid), dtype=np.float64)
            n_methods = 0
            for m in TM_METHODS:
                try:
                    tm = BinaryTsetlinMachine(clauses=TM_CLAUSES, T=TM_T, s=TM_S, epochs=TM_EPOCHS,
                                              boost_true_positive_feedback=TM_BOOST,
                                              max_included_literals=TM_MAX_INCLUDED)
                    tm.fit(Xtr_feats[m][mask], yv)
                    class_sums = tm.predict_class_sums(Xte_feats[m])[:, 1]  # positive-class votes
                    rng = np.ptp(class_sums)
                    if rng > 0:
                        votes += class_sums / rng        # composite: sum of range-normalized class sums
                        n_methods += 1
                except Exception as e:
                    print(f"  {lab}/{m}: TM failed ({e}), skipping method")
            if n_methods == 0:
                conf = np.full(len(te_lab_uid), default_rate)
            else:
                lo, hi = votes.min(), votes.max()
                conf = np.full(len(te_lab_uid), 0.5) if hi <= lo else (votes - lo) / (hi - lo)
            for pos, val in zip(te_lab_pos, conf):
                pred_tm.loc[pos, lab] = val
            print(f"  {lab}: composite over {n_methods} methods, {round(_time.time() - t0, 1)}s")

        TM_OK = True
        print("Composite TM predictions ready for", len(pred_tm), "test studies.")
else:
    print("Skipping TM image model (images are only mounted on Kaggle). Using CSV baselines.")


In [ ]:
# Pick the best model(s), then write the official submission.
if TM_OK and BASELINE_B_OK:
    pred = pred_tm.copy()
    pb = pred_test.loc[test["StudyInstanceUID"]].reset_index(drop=True)
    for lab in LABELS:
        if pred_tm[lab].nunique() <= 1:       # TM fell back to a constant for this label
            pred[lab] = pb[lab]
            continue
        auc_b = cv_scores.get(lab, np.nan)
        w_b = 0.0 if pd.isna(auc_b) else float(np.clip((auc_b - 0.5) * 2, 0, 0.5))
        r_tm = pred_tm[lab].rank(pct=True)
        r_b = pb[lab].rank(pct=True)
        blend = (1 - w_b) * r_tm + w_b * r_b
        mn, mx = float(blend.min()), float(blend.max())
        pred[lab] = (blend - mn) / (mx - mn + 1e-9)
    print("Using rank-blend of Composite TM + Baseline B (B weighted by its CV AUC).")
elif TM_OK:
    pred = pred_tm.copy()
    print("Using Composite TM (image model) only.")
elif BASELINE_B_OK:
    pred = pred_test.copy()
    pred.insert(0, "StudyInstanceUID", pred.index)
    pred = pred.reset_index(drop=True)
    print("Using Baseline B (series-descriptor logistic regression).")
else:
    pred = submission_a.copy()
    print("Using Baseline A (prevalence constant).")

# Enforce the exact column set and order of sample_submission.csv.
submission = pred[sample_sub.columns].copy()

assert list(submission.columns) == list(sample_sub.columns), "column mismatch"
assert len(submission) == len(test), "row-count mismatch"
assert submission["StudyInstanceUID"].is_unique, "duplicate StudyInstanceUID"

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(submission), "rows x", submission.shape[1], "cols.")
submission.head()


In [ ]:
# Quick check: submission.csv round-trips and is well-formed.
chk = pd.read_csv("submission.csv", dtype={"StudyInstanceUID": str})
print("round-trip ok:", chk.shape, "| labels in range [0,1]:", bool(chk[LABELS].min().min() >= 0 and chk[LABELS].max().max() <= 1))


## 8. Bonus - peek at a real DICOM slice (on Kaggle only, zero extra download)

On Kaggle the images are already mounted, so we can read one slice with `pydicom` for free.
Locally we skip this so the download stays minimal.


In [ ]:
if ON_KAGGLE:
    import glob as _glob
    import pydicom
    import matplotlib.pyplot as plt

    study_dirs = sorted(_glob.glob(os.path.join(DATA_DIR, "train_series", "*")))
    if study_dirs:
        series_dirs = sorted(_glob.glob(os.path.join(study_dirs[0], "*")))
        slices = sorted(_glob.glob(os.path.join(series_dirs[0], "*.dcm"))) if series_dirs else []
        if slices:
            mid = slices[len(slices) // 2]
            try:
                ds = pydicom.dcmread(mid, force=True)
                px = ds.pixel_array
                print("slice:", os.path.basename(mid))
                print("shape:", px.shape, "| BitsAllocated:", ds.BitsAllocated, "| Plane:", ds.get("ImagePositionPatient"))
                plt.figure(figsize=(3, 3))
                plt.imshow(px, cmap="gray")
                plt.axis("off")
                plt.show()
            except Exception as e:
                print("could not decode preview slice:", e)
        else:
            print("No slices found.")
    else:
        print("No training studies mounted.")
else:
    print("Skipping DICOM preview (not downloaded locally to keep the download small).")


## 9. Next steps (where this baseline goes from here)

1. **Reports as weak labels (implemented in section 5)** - extend the multilingual keyword lists and add an LLM-based miner for even cleaner pseudo-labels; `TM_N_TRAIN` can then be raised so the image model uses all ~5,000 studies.
2. **Images (v2 upgrades: 32px, plane-aware, +methods)** - the Composite TM is now plane-aware (per-finding preferred series via `TM_PLANE_OF_LABEL`), runs at `TM_IMG_SIZE=32`, and adds Sobel + thermometer feature maps. Next levers: raise `TM_N_TRAIN` / `TM_CLAUSES` / `TM_EPOCHS`, add HOG/color composites, or a per-plane fusion. A 3D/2.5D CNN on selected series (sagittal PD, sagittal fat-sat T2, coronal T1) is the standard strong alternative and would use a GPU accelerator.
3. **Series selection (partially implemented)** - v2 uses `train_series.csv` plane + fluid sensitivity; refine with per-finding fluid/fat-sat preference and score series importance from the section-6 descriptor baseline.
4. **Memory** - 570 GB of DICOMs cannot all be held in RAM; stream slices on the fly / preprocess to compact numpy volumes.
5. **Efficiency track** - this competition also rewards fast models (score/time); CPU-friendly TMs and logistic regression keep you competitive there.
